# Day 15: An Even Bigger Electric Potential Grid

#### &#9989; **Write your name here**

When solving the 80-by-120 grid in Day 14, there were 9600 variables representing the different electric potential values. The resulting system of equations pushes the limits of `np.linalg`, and can take several seconds to be solved. This begs the question, how can compute solutions to bigger and bigger grids without using up too many resources?

*Can we solve a 1000x1000 grid?*

<img src="https://raw.githubusercontent.com/pattihamerski/PH-36X-Public/main/images/one-million.gif"
     alt="one million values of electric potential -- a famous quote from an old movie"
     width="500"
/>

A 1000-by-1000 grid means $M=1000$, and $N=1000$. In total, this is **one million** values of electric potential contained in the grid.

If we try to run code using the methods from previous assignments, your kernel will probably die. Mine does. Feel free to run the cell below and try it out.

**&#9989; Task 0.1:** Read through the code below. Add comments to **explain what each part of the code is doing** (you choose where the comments go based on what you think needs to be explained).

In [ ]:
# add comments to this cell

# this code will kill your kernel
# we need new methods...

from scipy import linalg
import numpy as np

M = 1000
N = 1000

b = np.zeros(M * N)

b[:N] = 100
b[(M-1)*N:] = 50
b[0::N] = 10
b[N-1::N] = 20

L = np.zeros((M*N, M*N))

for i in range(M):
    for j in range(N):
        k = N*i + j

        if i == 0 or i == M-1 or j == 0 or j == N-1:
            L[k, k] = 1
        
        else:
            L[k, k] = 1
            L[k, k-1] = -1/4
            L[k, k+1] = -1/4
            L[k, k-N] = -1/4
            L[k, k+N] = -1/4

linalg.solve(L, b)

We can take advantage of the fact that the $\mathbb{L}$-matrix is mostly just a bunch of 0s. There are five nonzero diagonal strips in the whole matrix, which means that only a tiny portion of the matrix has nonzero elements. For the grid we are solving today, this fraction is about 0.000005.

This teensy weensy fraction means that our matrix is extremely **"sparse."** This word is reserved for matrices that are mostly filled with zeros. It can be useful to know this in computing because "sparse" matrices can be represented in terms of their nonzero elements, leaving out all the zeros that usually take up so much space.

Below, you can see an example $\mathbb{L}$-matrix (for the 4x4 grid) whose non-zero diagonals have been highlighted. Every element not shown is 0, and many of the elements on the diagonals are also 0!

We will focus our attention on these diagonals, because these are where all the non-zero information about the $\mathbb{L}$-matrix is stored.

<img src="https://raw.githubusercontent.com/pattihamerski/PH-36X-Public/main/images/diags-highlight.png"
     alt="Diagionals highlighted for the matrix you get from a 4x4 grid"
     width="500"
/>

---

# Part 1: Using the `sparse` package

Today, we use a new `scipy` package called `sparse`. Run the code below to load it in.

In [3]:
from scipy import sparse
import numpy as np
import matplotlib.pyplot as plt

We will solve our potential grid using sparse matrices to increase efficiency. We'll begin with a 4x4 grid, represented by the matrix in the example shown above, just before Part 1. 

Some diagrams to refresh how we have been setting up this matrix equation:

<img src="https://raw.githubusercontent.com/pattihamerski/PH-36X-Public/main/images/MN-grid-ijk.png"
     alt="MxN example grid, showing indexing with (i,j) and k"
     width="750"
/>

<img src="https://raw.githubusercontent.com/pattihamerski/PH-36X-Public/main/images/laplace-matrix-equation.png"
     alt="Matrix equation with values shown indicating the electric potential values in the electric potential grid"
     width="750"
/>

**&#9989; Task 1.1:** The boundary conditions of the 4x4 grid are: 25 V on top row, 10 V on the bottom row, 0 V on the sides.

Construct the b-vector for this 4x4 grid.

In [ ]:
# your answer here

The `sparse` package requires each diagonal to be defined prior to constructing a matrix. See below for a visual of each diagonal.

<img src="https://raw.githubusercontent.com/pattihamerski/PH-36X-Public/main/images/diags-highlight.png"
     alt="Diagionals highlighted for the matrix you get from a 4x4 grid"
     width="500"
/>

**&#9989; Task 1.2:** Create each diagonal as a 1D numpy array.

*Note, not all diagonals have the same length.*

In [ ]:
# your answer here

The `sparse` package has its own function for constructing a matrix from diagonals: `sparse.diags_array`.

You can call this function like `sparse.diags_array(diags, offsets=[nums])`, where `diags` is a list of your diagonal arrays, and `offsets` is a list of integers indicating how far offset each diagonal is from the main diagonal (negative for below, positive for above). Note: `offsets` must be named explicitly during the function call.

**&#9989; Task 1.3:** Create a sparse matrix from your diagonals.

In [37]:
# your answer here

Lastly, there is `sparse.linalg.spsolve`, which works the same way as `linalg.solve`, except that the $\mathbb{L}$-matrix is sparse.

**&#9989; Task 1.4:** Solve your sparse matrix equation by plugging in your b-vector and your sparse $\mathbb{L}$-matrix into the sparse solver. Print the result and verify it makes sense based on your initial conditions.

*Note: You will receive a warning about the matrix type when you run the solver. This is mostly okay to ignore (efficiency-wise, it only makes about 5% difference for this problem). To put your matrix in a "compressed sparse row" format like the warning requests, you can input `L.tocsr()` into the function instead of just `L`, or you can add `format="csr"` into the function call in the previous task.*

In [ ]:
# your answer here

This method saves more and more space the larger your matrix is. For the 4x4 grid example, this isn't much of a difference. Below, we'll apply the sparse solving method to a much bigger matrix!

<img src="https://raw.githubusercontent.com/pattihamerski/PH-36X-Public/main/images/jaws-matrix.gif"
     alt="You're gonna need a bigger matrix"
     width="500"
/>

---

# Part 2: A bigger matrix

**&#9989; Task 2:** In this part, your only task is to **solve the 1000x1000 electric potential grid using sparse matrices.** You can choose any boundary conditions as long as they are not all the same value.

*Here are some tips/hints you may find helpful:*
- *The `np.tile` function can be useful for creating repeated sequences within an array*
- *Loops are not needed to solve this, but there is no shame in using them*
- *Use the visualization code further below to see how your solution turns out*
- *Even when coded optimally, a solution still takes at least 30 seconds to run. Start smaller, like 200x200, and work your way up after you've got your code working the way you want.*
- *Extra: if you have your code fully working, try a non-square grid ($N \ne M$) to see whether your implementation holds up and make it more generalizable.*

In [60]:
# your answer here

In [1]:
# use this code chunk however you like

reshaped = v.reshape(1000, 1000)

plt.imshow(reshaped, cmap='turbo')
plt.title("the big potential grid!")
plt.xlabel("x")
plt.ylabel("y")
plt.colorbar()
plt.show()

NameError: name 'v' is not defined